In [ ]:
from typing import Annotated, TypedDict
import operator
import pandas as pd
import re

from langchain_core.messages import HumanMessage, AIMessage
from langchain_core.tools import tool
from langchain_openai import ChatOpenAI
from langgraph.graph import StateGraph, START, END
from langgraph.graph.message import add_messages
from langgraph.prebuilt import ToolNode, tools_condition

import os 
from dotenv import load_dotenv

load_dotenv()

OPENAI_API_KEY = os.getenv("OPENAI_API_KEY")
LANGSMITH_API_KEY = os.getenv("LANGSMITH_API_KEY")


print(os.getenv("LANGSMITH_TRACING"))
print(os.getenv("LANGSMITH_PROJECT"))
print(bool(os.getenv("LANGSMITH_API_KEY")))
print(os.getenv(""))
# ---------- Tools ----------

@tool
def calculator(a: float, b: float, operation: str) -> float:
    """Perform a basic arithmetic calculation on two numbers.

    Use this when the user asks for a math calculation involving two numbers,
    such as multiplying seats by price, or adding/subtracting amounts.

    Args:
        a: The first number.
        b: The second number.
        operation: One of 'add', 'subtract', 'multiply', 'divide'.

    Returns:
        The numeric result of applying the operation to a and b.
    """
    if operation == "add":
        return a + b
    elif operation == "subtract":
        return a - b
    elif operation == "multiply":
        return a * b
    elif operation == "divide":
        return a / b
    else:
        raise ValueError(f"Unsupported operation: {operation}")


@tool
def lookup_price(plan: str) -> str:
    """Look up pricing information for a given subscription plan.

    Use this when the user asks how much a plan costs, or asks about
    seat pricing or bulk discounts for a specific plan name.

    Args:
        plan: The plan name to look up, e.g. 'Basic', 'Pro', or 'Enterprise'.

    Returns:
        A string summarizing the plan's price per seat, minimum seats
        required for a discount, and the discount percentage.
    """
    df = pd.read_excel("/Users/nitishledalla/Desktop/projects/support_assistant/data/pricing.xlsx")
    row = df[df["plan"].str.lower() == plan.lower()]

    if row.empty:
        return f"No pricing found for plan '{plan}'."

    row = row.iloc[0]
    return (
        f"Plan: {row['plan']}, "
        f"Price per seat: ${row['price_per_seat']}, "
        f"Min seats for discount: {row['min_seats_for_discount']}, "
        f"Discount: {row['discount_pct']}%"
    )


@tool
def lookup_policy(policy_id: str) -> str:
    """Look up the full text of a company policy by its ID.

    Use this when the user asks about a specific policy, refund rules,
    cancellation terms, or anything referencing a policy ID like 'P-001'.

    Args:
        policy_id: The policy ID to look up, e.g. 'P-001', 'P-003'.

    Returns:
        The full text of the matching policy, or a message if not found.
    """
    file_path = "/Users/nitishledalla/Desktop/projects/support_assistant/data/policies.txt"
    with open(file_path, "r") as f:
        content = f.read()

    pattern = rf'({re.escape(policy_id)} \| .*?)(?=\nP-\d+ \||\Z)'
    match = re.search(pattern, content, re.DOTALL)

    if match:
        return match.group(1).strip()
    else:
        return f"No policy found for ID '{policy_id}'."


# ---------- Model ----------
llm = ChatOpenAI(model="gpt-4")
llm_with_tools = llm.bind_tools([calculator, lookup_policy, lookup_price])


# ---------- State ----------
class InputState(TypedDict):
    question: str


class OutputState(TypedDict):
    answer: str


class State(TypedDict):
    question: str
    answer: str
    messages: Annotated[list, add_messages]
    notes: Annotated[list, operator.add]
    run: int


# ---------- Nodes ----------
def normalize(state: State):
    tmp = state['question']
    tmp = tmp.lower().strip()
    return {"messages": [HumanMessage(content=tmp)]}


def agent(state: State):
    current_run = state.get('run', 0)

    if current_run >= 5:
        return {"messages": [], "run": current_run}

    response = llm_with_tools.invoke(state['messages'])

    update = {"messages": [response], "run": current_run + 1}
    if response.content:
        update["answer"] = response.content

    return update


tools_node = ToolNode([calculator, lookup_policy, lookup_price])


# ---------- Graph ----------
builder = StateGraph(State, input_schema=InputState, output_schema=OutputState)

builder.add_node("normalize", normalize)
builder.add_node("agent", agent)
builder.add_node("tools", tools_node)

builder.add_edge(START, "normalize")
builder.add_edge("normalize", "agent")
builder.add_conditional_edges("agent", tools_condition)
builder.add_edge("tools", "agent")

graph = builder.compile()

result = graph.invoke({"question": "What is the price of the Pro plan?"})
print(result)

true
support-assistant
True
None
{'answer': 'The Pro Plan is priced at $80 per seat. If you have 15 or more seats, you can avail a discount of 0.1% on your total cost.'}


In [2]:
from typing import Annotated, TypedDict
import operator
import pandas as pd
import re

from langchain_core.messages import HumanMessage, AIMessage
from langchain_core.tools import tool
from langchain_openai import ChatOpenAI
from langgraph.graph import StateGraph, START, END
from langgraph.graph.message import add_messages
from langgraph.prebuilt import ToolNode, tools_condition


# ---------- Tools ----------

@tool
def calculator(a: float, b: float, operation: str) -> float:
    """Perform a basic arithmetic calculation on two numbers.

    Use this when the user asks for a math calculation involving two numbers,
    such as multiplying seats by price, or adding/subtracting amounts.

    Args:
        a: The first number.
        b: The second number.
        operation: One of 'add', 'subtract', 'multiply', 'divide'.

    Returns:
        The numeric result of applying the operation to a and b.
    """
    if operation == "add":
        return a + b
    elif operation == "subtract":
        return a - b
    elif operation == "multiply":
        return a * b
    elif operation == "divide":
        return a / b
    else:
        raise ValueError(f"Unsupported operation: {operation}")


@tool
def lookup_price(plan: str) -> str:
    """Look up pricing information for a given subscription plan.

    Use this when the user asks how much a plan costs, or asks about
    seat pricing or bulk discounts for a specific plan name.

    Args:
        plan: The plan name to look up, e.g. 'Basic', 'Pro', or 'Enterprise'.

    Returns:
        A string summarizing the plan's price per seat, minimum seats
        required for a discount, and the discount percentage.
    """
    df = pd.read_excel("/Users/nitishledalla/Desktop/projects/support_assistant/data/pricing.xlsx")
    row = df[df["plan"].str.lower() == plan.lower()]

    if row.empty:
        return f"No pricing found for plan '{plan}'."

    row = row.iloc[0]
    return (
        f"Plan: {row['plan']}, "
        f"Price per seat: ${row['price_per_seat']}, "
        f"Min seats for discount: {row['min_seats_for_discount']}, "
        f"Discount: {row['discount_pct']}%"
    )


@tool
def lookup_policy(policy_id: str) -> str:
    """Look up the full text of a company policy by its ID.

    Use this when the user asks about a specific policy, refund rules,
    cancellation terms, or anything referencing a policy ID like 'P-001'.

    Args:
        policy_id: The policy ID to look up, e.g. 'P-001', 'P-003'.

    Returns:
        The full text of the matching policy, or a message if not found.
    """
    file_path = "/Users/nitishledalla/Desktop/projects/support_assistant/data/policies.txt"
    with open(file_path, "r") as f:
        content = f.read()

    pattern = rf'({re.escape(policy_id)} \| .*?)(?=\nP-\d+ \||\Z)'
    match = re.search(pattern, content, re.DOTALL)

    if match:
        return match.group(1).strip()
    else:
        return f"No policy found for ID '{policy_id}'."


# ---------- Model ----------

llm = ChatOpenAI(model="gpt-4")
llm_with_tools = llm.bind_tools([calculator, lookup_policy, lookup_price])


# ---------- State ----------

class InputState(TypedDict):
    question: str


class OutputState(TypedDict):
    answer: str


class State(TypedDict):
    question: str
    answer: str
    messages: Annotated[list, add_messages]
    notes: Annotated[list, operator.add]
    run: int


# ---------- Helpers for clean trace printing ----------

def section(title: str):
    print()
    print("=" * 70)
    print(title)
    print("=" * 70)


def show_state(state: State):
    for k, v in state.items():
        print(f"    {k!r:12} -> {v!r}")


# ---------- Nodes ----------

def normalize(state: State):
    section("[normalize] INPUT")
    print("type(state):", type(state))
    show_state(state)

    tmp = state['question']
    tmp = tmp.lower().strip()
    print(f"\nnormalized question -> {tmp!r}")

    output = {"messages": [HumanMessage(content=tmp)]}

    section("[normalize] OUTPUT (partial update)")
    show_state(output)

    return output


def agent(state: State):
    section("[agent] INPUT")
    print("type(state):", type(state))
    show_state(state)

    current_run = state.get('run', 0)
    print(f"\ncurrent run count -> {current_run}")

    if current_run >= 5:
        print("!! run cap reached, stopping without another LLM call !!")
        return {"messages": [], "run": current_run}

    response = llm_with_tools.invoke(state['messages'])

    section("[agent] raw LLM response")
    print("type(response):", type(response))
    print("response.content   :", repr(response.content))
    print("response.tool_calls:", response.tool_calls)

    update = {"messages": [response], "run": current_run + 1}
    if response.content:
        update["answer"] = response.content

    section("[agent] OUTPUT (partial update)")
    show_state(update)

    return update


tools_node = ToolNode([calculator, lookup_policy, lookup_price])


# ---------- Graph ----------

builder = StateGraph(State, input_schema=InputState, output_schema=OutputState)

builder.add_node("normalize", normalize)
builder.add_node("agent", agent)
builder.add_node("tools", tools_node)

builder.add_edge(START, "normalize")
builder.add_edge("normalize", "agent")
builder.add_conditional_edges("agent", tools_condition)
builder.add_edge("tools", "agent")

graph = builder.compile()

section("BEFORE invoke")
input_payload = {"question": "What is the price of the Pro plan?"}
print("type(input_payload):", type(input_payload))
print("input_payload      :", input_payload)

result = graph.invoke(input_payload)

section("AFTER invoke (final state, filtered to OutputState)")
print("type(result):", type(result))
print("result      :", result)


BEFORE invoke
type(input_payload): <class 'dict'>
input_payload      : {'question': 'What is the price of the Pro plan?'}

[normalize] INPUT
type(state): <class 'dict'>
    'question'   -> 'What is the price of the Pro plan?'
    'messages'   -> []
    'notes'      -> []

normalized question -> 'what is the price of the pro plan?'

[normalize] OUTPUT (partial update)
    'messages'   -> [HumanMessage(content='what is the price of the pro plan?', additional_kwargs={}, response_metadata={})]

[agent] INPUT
type(state): <class 'dict'>
    'question'   -> 'What is the price of the Pro plan?'
    'messages'   -> [HumanMessage(content='what is the price of the pro plan?', additional_kwargs={}, response_metadata={}, id='6da08f88-a83c-4239-a359-23504aaaabdf')]
    'notes'      -> []

current run count -> 0

[agent] raw LLM response
type(response): <class 'langchain_core.messages.ai.AIMessage'>
response.content   : ''
response.tool_calls: [{'name': 'lookup_price', 'args': {'plan': 'Pro'}, 'id'